In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_E.xpt"
csv_file = DATA_DIR / "GLU_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched XPT values for fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 GLU_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI",
    "PHAFSTHR",
    "PHAFSTMN"
]


if df.shape != (3315, 8):

    raise ValueError(
        f"Unexpected GLU_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GLU_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: GLU_E structure = "
    "3,315 rows x 8 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# CRITICAL:
#
# Keep the actual LAB MEASUREMENTS as source-decoded floats.
#
# DO NOT:
#
# - convert measurement fields to Int64
# - round()
# - use float_format
# - trim legitimate .0
# - recalculate derived SI fields
#
# Even LBXGLU must remain a measurement float although
# all released values happen to be mathematically whole.
#

MEASUREMENT_COLUMNS = [
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "PHAFSTHR",
    "PHAFSTMN"
]


print(
    "\nLab/weight measurement fields preserved AS IS:"
)

for col in MEASUREMENT_COLUMNS:
    print(" ", col)


print(
    "\nIdentifier/time fields written as clean integers:"
)

for col in INTEGER_COLUMNS:
    print(" ", col)


# ============================================================
# 5. KNOWN SAS/XPORT TINY ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)


print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)


print(
    "Tiny artifacts:",
    f"{int(tiny_counts.sum()):,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


if int(
    true_zero_counts.sum()
) != 0:

    raise ValueError(
        "Unexpected true numeric zeros "
        "already present in raw GLU_E XPT."
    )


# ============================================================
# 6. EXPECTED TINY COUNTS
#    FROM ATTACHED GLU_E XPT
# ============================================================

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 515,
    "PHAFSTHR": 61,
    "PHAFSTMN": 61
}


EXPECTED_TINY_TOTAL = 637


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    missing_columns = (
        set(EXPECTED_TINY_COUNTS)
        -
        set(actual_tiny_counts)
    )


    unexpected_columns = (
        set(actual_tiny_counts)
        -
        set(EXPECTED_TINY_COUNTS)
    )


    different_counts = {
        col: {
            "expected": EXPECTED_TINY_COUNTS.get(col),
            "actual": actual_tiny_counts.get(col)
        }
        for col in (
            set(EXPECTED_TINY_COUNTS)
            &
            set(actual_tiny_counts)
        )
        if (
            EXPECTED_TINY_COUNTS[col]
            !=
            actual_tiny_counts[col]
        )
    }


    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Missing variables: {sorted(missing_columns)}\n"
        f"Unexpected variables: {sorted(unexpected_columns)}\n"
        f"Different counts: {different_counts}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


print(
    "\nPASS: Exactly 637 tiny artifacts "
    "found in the expected 3 variables."
)


# ============================================================
# 7. SAFETY CHECK:
#    ONLY APPROVED VARIABLES MAY CONTAIN TINY VALUES
# ============================================================

APPROVED_ZERO_COLUMNS = {
    "WTSAF2YR",
    "PHAFSTHR",
    "PHAFSTMN"
}


unexpected_tiny_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected variables:\n"
        + str(
            sorted(
                unexpected_tiny_columns
            )
        )
    )


print(
    "PASS: Tiny artifacts occur only "
    "in validated zero-permitted fields."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print(
    "\n--- ZERO RESTORATION ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 637 validated zeros restored."
)


# ============================================================
# 9. VERIFY ZERO COUNTS
# ============================================================

for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {actual_zero_count:,}."
        )


print(
    "PASS: Final restored-zero counts are exact."
)


# ============================================================
# 10. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SEQN CHECK
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41479:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN = 41479-51623, "
    "no missing, no duplicates."
)


# ============================================================
# 12. CDC CHECK - WTSAF2YR
# ============================================================

#
# CDC:
#
# WTSAF2YR
# Valid   = 3,315
# Missing = 0
# Range   = 0 to 404655.623
#
# The 515 tiny source values are therefore
# validated zero weights.
#

s = df["WTSAF2YR"]


if int(
    s.notna()
    .sum()
) != 3315:

    raise ValueError(
        "WTSAF2YR valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 0:

    raise ValueError(
        "WTSAF2YR missing count differs."
    )


if s.min() != 0:

    raise ValueError(
        f"Unexpected WTSAF2YR minimum: {s.min()}"
    )


if not np.isclose(
    s.max(),
    404655.623,
    rtol=0,
    atol=1e-12
):

    raise ValueError(
        f"Unexpected WTSAF2YR maximum: {s.max()}"
    )


if int(
    s.eq(0)
    .sum()
) != 515:

    raise ValueError(
        "WTSAF2YR zero count differs."
    )


print(
    "\nPASS: WTSAF2YR"
)

print(
    "      Valid   = 3,315"
)

print(
    "      Missing = 0"
)

print(
    "      Range   = 0-404655.623"
)

print(
    "      Zeros   = 515"
)


# ============================================================
# 13. CDC CHECK - LBXGLU
# ============================================================

#
# Fasting Glucose (mg/dL)
#
# CDC:
# Valid   = 3,110
# Missing = 205
# Range   = 38-584
#
# KEEP AS LAB MEASUREMENT FLOAT.
#

s = df["LBXGLU"]


if int(
    s.notna()
    .sum()
) != 3110:

    raise ValueError(
        "LBXGLU valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 205:

    raise ValueError(
        "LBXGLU missing count differs."
    )


if s.min() != 38:

    raise ValueError(
        "Unexpected LBXGLU minimum."
    )


if s.max() != 584:

    raise ValueError(
        "Unexpected LBXGLU maximum."
    )


print(
    "\nPASS: LBXGLU"
)

print(
    "      Valid   = 3,110"
)

print(
    "      Missing = 205"
)

print(
    "      Range   = 38-584"
)


# ============================================================
# 14. CDC CHECK - LBDGLUSI
# ============================================================

#
# CDC-derived glucose SI value.
#
# DO NOT recompute it.
# Preserve released XPT value.
#

s = df["LBDGLUSI"]


if int(
    s.notna()
    .sum()
) != 3110:

    raise ValueError(
        "LBDGLUSI valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 205:

    raise ValueError(
        "LBDGLUSI missing count differs."
    )


if not np.isclose(
    s.min(),
    2.109,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDGLUSI minimum."
    )


if not np.isclose(
    s.max(),
    32.418,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDGLUSI maximum."
    )


print(
    "PASS: LBDGLUSI = 2.109-32.418."
)


# ============================================================
# 15. CDC CHECK - LBXIN
# ============================================================

s = df["LBXIN"]


if int(
    s.notna()
    .sum()
) != 3071:

    raise ValueError(
        "LBXIN valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 244:

    raise ValueError(
        "LBXIN missing count differs."
    )


if not np.isclose(
    s.min(),
    1.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBXIN minimum."
    )


if not np.isclose(
    s.max(),
    231.67,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBXIN maximum."
    )


print(
    "PASS: LBXIN = 1-231.67."
)


# ============================================================
# 16. CDC CHECK - LBDINSI
# ============================================================

#
# CDC-derived insulin SI value.
#
# DO NOT recompute.
#

s = df["LBDINSI"]


if int(
    s.notna()
    .sum()
) != 3071:

    raise ValueError(
        "LBDINSI valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 244:

    raise ValueError(
        "LBDINSI missing count differs."
    )


if not np.isclose(
    s.min(),
    6.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDINSI minimum."
    )


if not np.isclose(
    s.max(),
    1390.02,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDINSI maximum."
    )


print(
    "PASS: LBDINSI = 6-1390.02."
)


# ============================================================
# 17. CDC CHECK - FASTING TIME
# ============================================================

#
# PHAFSTHR:
# Valid   = 3,223
# Missing = 92
# Range   = 0-40
#
# PHAFSTMN:
# Valid   = 3,223
# Missing = 92
# Range   = 0-59
#

s = df["PHAFSTHR"]


if not (
    int(s.notna().sum()) == 3223
    and
    int(s.isna().sum()) == 92
    and
    s.min() == 0
    and
    s.max() == 40
    and
    int(s.eq(0).sum()) == 61
):

    raise ValueError(
        "PHAFSTHR differs from expected "
        "CDC/XPT values."
    )


print(
    "\nPASS: PHAFSTHR"
)

print(
    "      Valid   = 3,223"
)

print(
    "      Missing = 92"
)

print(
    "      Range   = 0-40"
)

print(
    "      Zeros   = 61"
)


s = df["PHAFSTMN"]


if not (
    int(s.notna().sum()) == 3223
    and
    int(s.isna().sum()) == 92
    and
    s.min() == 0
    and
    s.max() == 59
    and
    int(s.eq(0).sum()) == 61
):

    raise ValueError(
        "PHAFSTMN differs from expected "
        "CDC/XPT values."
    )


print(
    "PASS: PHAFSTMN"
)

print(
    "      Valid   = 3,223"
)

print(
    "      Missing = 92"
)

print(
    "      Range   = 0-59"
)

print(
    "      Zeros   = 61"
)


# ============================================================
# 18. FRACTIONAL VALUE CHECK
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- GENUINE FRACTIONAL VALUES ---"
)


for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "WTSAF2YR": 2799,
    "LBDGLUSI": 3110,
    "LBXIN": 2992,
    "LBDINSI": 2961
}


if fractional_columns != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected genuine fractional-value pattern.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COUNTS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: Genuine fractional fields match "
    "the attached XPT."
)


# ============================================================
# 19. VERIFY INTEGER FIELDS ARE WHOLE NUMBERS
# ============================================================

integer_fraction_failures = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        integer_fraction_failures[
            col
        ] = fractional_count


if integer_fraction_failures:

    raise ValueError(
        "STOP: Integer-designated fields "
        "contain fractional values:\n"
        + str(
            integer_fraction_failures
        )
    )


print(
    "PASS: SEQN and fasting-time fields "
    "are whole-number values."
)


# ============================================================
# 20. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY:
#
# WTSAF2YR tiny -> 0.0
# PHAFSTHR tiny -> 0
# PHAFSTMN tiny -> 0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. CLEAN ONLY INTEGER FIELDS
# ============================================================

#
# SEQN:
#
# 41479.0 -> 41479
#
#
# PHAFSTHR:
#
# 12.0 -> 12
# 0.0  -> 0
#
#
# PHAFSTMN:
#
# 30.0 -> 30
# 0.0  -> 0
#
#
# DO NOT convert laboratory measurements to Int64.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN / fasting-time fields "
    "converted to clean integer formatting."
)

print(
    "PASS: All laboratory/weight measurements "
    "remain source-decoded floats."
)


# ============================================================
# 22. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in INTEGER_COLUMNS:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "\nPASS: Pre-export values equal expected XPT "
    "after ONLY validated zero restorations."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# recalculation
# decimal quantization
#
# Let pandas write the actual decoded source values.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 24. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 26. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 8 columns numerically match "
    "the expected XPT exactly."
)


# ============================================================
# 27. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 28. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 3315:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 29. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# Integer fields:
#
# str(int(source_value))
#
#
# Measurement fields:
#
# str(float(source_value))
#
# Examples:
#
# SEQN:
# 41479.0 -> "41479"
#
# LBXGLU:
# 92.0 -> "92.0"
#
# LBDGLUSI:
# 5.106 -> "5.106"
#
# LBXIN:
# 7.0 -> "7.0"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "GLU_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 30. UNWANTED .0 CHECK
#     INTEGER FIELDS ONLY
# ============================================================

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer-field '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 remains in "
        "SEQN/fasting-time fields."
    )


print(
    "PASS: No unwanted .0 in integer fields."
)


# ============================================================
# 31. STRICT LAB MEASUREMENT RAW CHECK
# ============================================================

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Lab-measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Laboratory measurement representation changed."
    )


print(
    "PASS: Every lab/weight measurement "
    "is preserved AS IS."
)


# ============================================================
# 32. LEGITIMATE MEASUREMENT .0 CHECK
# ============================================================

#
# These counts come from the actual attached source
# after the validated tiny -> zero correction.
#

EXPECTED_MEASUREMENT_DOT_ZERO = {
    "WTSAF2YR": 516,
    "LBXGLU": 3110,
    "LBDGLUSI": 0,
    "LBXIN": 79,
    "LBDINSI": 110
}


print(
    "\n--- LEGITIMATE MEASUREMENT .0 CHECK ---"
)


for col, expected_count in (
    EXPECTED_MEASUREMENT_DOT_ZERO.items()
):

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate .0 count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate lab/weight .0 "
    "representations preserved."
)


# ============================================================
# 33. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(scientific_examples) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 34. FINAL TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains in CSV."
)


# ============================================================
# 35. FINAL ZERO COUNT CHECK
# ============================================================

zero_failures = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        zero_failures[col] = {
            "expected": expected_count,
            "actual": actual_count
        }


if zero_failures:

    raise ValueError(
        "Final zero counts changed:\n"
        + str(
            zero_failures
        )
    )


print(
    "PASS: All 637 restored zero values "
    "remain correct after CSV export."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GLU_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41479-51623"
)


print(
    "\nValidated tiny -> zero corrections:"
)

print(
    "WTSAF2YR = 515"
)

print(
    "PHAFSTHR = 61"
)

print(
    "PHAFSTMN = 61"
)

print(
    "TOTAL     = 637"
)


print(
    "\nLab/weight measurements preserved AS IS:"
)

for col in MEASUREMENT_COLUMNS:
    print(col)


print(
    "\nGenuine fractional variables:"
)

for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted integer .0:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "GLU_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 GLU_E XPT -> CSV
Rows: 3,315
Columns: 8
PASS: GLU_E structure = 3,315 rows x 8 columns.

Lab/weight measurement fields preserved AS IS:
  WTSAF2YR
  LBXGLU
  LBDGLUSI
  LBXIN
  LBDINSI

Identifier/time fields written as clean integers:
  SEQN
  PHAFSTHR
  PHAFSTMN

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 637

Tiny values by variable:
WTSAF2YR    515
PHAFSTHR     61
PHAFSTMN     61

PASS: Exactly 637 tiny artifacts found in the expected 3 variables.
PASS: Tiny artifacts occur only in validated zero-permitted fields.

--- ZERO RESTORATION ---
WTSAF2YR  : 515 tiny -> 0
PHAFSTHR  : 61 tiny -> 0
PHAFSTMN  : 61 tiny -> 0
PASS: All 637 validated zeros restored.
PASS: Final restored-zero counts are exact.
Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

--- SEQN CHECK ---
PASS: SEQN = 41479-51623, no missing, no duplicates.

PASS: WTSAF2YR
      Valid   = 3,315
      Missing = 0
      Range   = 0-404655.623
      Zeros   = 515

PASS: L